In [21]:
import tkinter as tk
from tkinter import ttk,messagebox as msgbox
from PIL import Image, ImageTk
import psycopg2
import json
import os

BG_DARK   = "#0D1B12"
BG_CARD   = "#152318"
ACCENT1   = "#39D98A"
ACCENT2   = "#F4A823"
ACCENT3   = "#3ddc6a"
ACCENT4   = "#27a34a"
TEXT_MAIN = "#E8F5EE"
TEXT_MUTE = "#7DA98A"
DANGER    = "#E05252"
BG        = "#0a0f0a"
TEXT      = "#e0f0e0"
MUTED     = "#5a7a5a"
ENTRY_BG  = "#111f11"
QUOTE     = '"Every item recycled is armour against digital waste"'

items = {
    "Phone"          ,
    "Laptop"          ,
    "Battery"        ,
    "Charger"         ,
    "Cable"           ,
    "Motherboard"    ,
    "Small Appliance" ,
    "Other"           
}

item_types = list(items)

class EcoTrackEnhancer:
    BG       = BG
    TEXT     = TEXT
    MUTED    = MUTED
    ENTRY_BG = ENTRY_BG
    ACCENT   = ACCENT1
    ACCENT2  = ACCENT2
    ACCENT3  = ACCENT3
    ACCENT4  = ACCENT4
    QUOTE    = QUOTE

    def __init__(self, root):
        self.root = root
        root.configure(bg=self.BG)

    def apply(self, widgets, show_quote=False, quote_y=310):
        styles = {
            "sidebars":  dict(bg=self.ACCENT4),
            "accent":    dict(bg=self.BG, fg=self.ACCENT),
            "muted":     dict(bg=self.BG, fg=self.MUTED),
            "entries":   dict(bg=self.ENTRY_BG, fg=self.TEXT, relief="flat",
                              insertbackground=self.ACCENT3, highlightthickness=1,
                              highlightbackground=self.ACCENT4),
            "buttons":   dict(bg=self.ACCENT4, fg=self.BG, relief="flat", bd=0),
            "secondary": dict(bg=self.BG, fg=self.ACCENT, relief="flat", bd=1),
            "dividers":  dict(bg=self.BG, highlightthickness=0),
        }
        for key, cfg in styles.items():
            for w in widgets.get(key, []):
                w.configure(**cfg)
                if key == "dividers":
                    w.delete("all")
                    w.create_line(10, 10, 10, 170, fill=self.ACCENT2, width=1)
        self.root.configure(bg=self.BG)
        if show_quote:
            tk.Label(self.root, text=self.QUOTE, font=("Arial", 7, "italic"),fg=self.ACCENT4, bg=self.BG, wraplength=380).place(x=210, y=quote_y)


class GoalTracker:
    def __init__(self):
        self.file = "ecotrack_goals.json"
        self.load()

    def load(self):
        if os.path.exists(self.file):
            with open(self.file, 'r') as f:
                d = json.load(f)
                self.daily, self.weekly, self.total, self.today = (
                    d.get('daily', 10), d.get('weekly', 50),
                    d.get('total', 0), d.get('today', 0)
                )
        else:
            self.daily, self.weekly, self.total, self.today = 10, 50, 0, 0

    def save(self):
        with open(self.file, 'w') as f:
            json.dump({'daily': self.daily, 'weekly': self.weekly,
                       'total': self.total, 'today': self.today}, f)

    def add(self, qty):
        self.today += qty
        self.total += qty
        self.save()
        msgs = []
        if self.today >= self.daily:
            msgs.append(f" Daily goal reached! ({self.today}/{self.daily})")
        if self.today >= self.weekly:
            msgs.append(f" Weekly goal reached! ({self.today}/{self.weekly})")
        return msgs

    def set_daily(self, amt): self.daily = amt; self.save()
    def set_weekly(self, amt): self.weekly = amt; self.save()


class EcoTrack:
    def __init__(self, root):
        self.root = root
        self.root.title("EcoTrack")
        self.root.geometry("600x400")
        self.root.resizable(False, False)
        self.root.configure(bg=BG)
        self.current_user = ""

        self.enhancer = EcoTrackEnhancer(self.root)
        self.goal = GoalTracker()

        self.star_rating = tk.IntVar(value=0)
        self.star_buttons = []

        img = Image.open("rec.png").resize((90, 90)) 
        self.photo = ImageTk.PhotoImage(img)

        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)
        self.create_tables()
        self.show_home()


    def connect_db(self):
        conn = psycopg2.connect(
            host="localhost",
            database="ewaste_db",
            user="postgres",
            password="Ugowashere2006!",
            port="5432"
        )
        return conn, conn.cursor()

    def create_tables(self):
        conn, cursor = self.connect_db()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS users(
                id SERIAL PRIMARY KEY,
                username VARCHAR(100) UNIQUE NOT NULL,
                password VARCHAR(100) NOT NULL,
                hint VARCHAR(200)
            )
        """)
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS waste_logs(
                id SERIAL PRIMARY KEY,
                username VARCHAR(100) NOT NULL,
                item VARCHAR(100) NOT NULL,
                quantity INTEGER NOT NULL
            )
        """)
        conn.commit()
        conn.close()

    def save_user(self, username, password, hint=""):
        conn, cursor = self.connect_db()
        try:
            cursor.execute("""
                INSERT INTO users(username, password, hint)
                VALUES (%s, %s, %s)
            """, (username, password, hint))
            conn.commit()
            return True
        except psycopg2.errors.UniqueViolation:
            conn.rollback()
            return False
        finally:
            conn.close()

    def table(self):
        style = ttk.Style()
        style.theme_use("clam")
        style.configure(
            "Treeview",
            background=BG_CARD,
            foreground=TEXT_MAIN,
            fieldbackground=BG_CARD,
            rowheight=30,
            font=("Arial", 10),
            bordercolor=ACCENT1,
            borderwidth=1
        )
        style.configure(
            "Treeview.Heading",
            background=ACCENT1,
            foreground=BG_DARK,
            font=("Arial", 10, "bold"),
            relief="flat"
        )
        style.map(
            "Treeview",
            background=[("selected", ACCENT4)],
            foreground=[("selected", "white")]
        )
        conn, cursor = self.connect_db()

        table = ttk.Treeview(
            self.root,
            columns=("ID", "USERNAME", "ITEM", "QTY"),
            show="headings",
            height=7
        )
        
        table.heading("ID", text="ID")
        table.heading("USERNAME", text="USERNAME")
        table.heading("ITEM", text="ITEM")
        table.heading("QTY", text="QTY")
    
        table.column("ID", width=60, anchor="center")
        table.column("USERNAME", width=140, anchor="center")
        table.column("ITEM", width=160, anchor="center")
        table.column("QTY", width=80, anchor="center")
    
        cursor.execute("""SELECT id, username, item, quantity FROM waste_logs""")
        records = cursor.fetchall()

        for row in records:
            table.insert("", tk.END, values=row)
        table.place(x=110, y=125)

        scrollbar = ttk.Scrollbar(
            self.root,
            orient="vertical",
            command=table.yview
        )
    
        table.configure(yscrollcommand=scrollbar.set)
    
        scrollbar.place(x=552, y=125, height=235)
    
        conn.close()
        
    def check_login(self, username, password):
        conn, cursor = self.connect_db()
        cursor.execute("""
            SELECT * FROM users WHERE username=%s AND password=%s
        """, (username, password))
        user = cursor.fetchone()
        conn.close()
        return user

    def get_user(self, username):
        conn, cursor = self.connect_db()
        cursor.execute("SELECT * FROM users WHERE username=%s", (username,))
        user = cursor.fetchone()
        conn.close()
        return user

    def update_password(self, username, new_password):
        conn, cursor = self.connect_db()
        cursor.execute("UPDATE users SET password=%s WHERE username=%s", (new_password, username))
        conn.commit()
        conn.close()

    def save_log(self, item, quantity):
        conn, cursor = self.connect_db()
        cursor.execute("""
            INSERT INTO waste_logs(username, item, quantity)
            VALUES (%s, %s, %s)
        """, (self.current_user, item, quantity))
        conn.commit()
        conn.close()


    def clear_page(self):
        for widget in self.root.winfo_children():
            widget.destroy()
        self.star_buttons = []

    def on_closing(self):
        if msgbox.askyesno("Quit", "Are you sure you want to exit?"):
            self.root.destroy()

       

    def build_header(self, subtitle="Online Waste Tracker"):
        left_frame = tk.Frame(self.root, bg="green", width=50, height=400)
        left_frame.place(x=50, y=0)
        
        image_label = tk.Label(self.root, image=self.photo)
        image_label.place(x=135, y=35)

        name_label = tk.Label(self.root, text="EcoTrack", fg="green",
                               font=("Arial", 37, "bold"))
        name_label.place(x=230, y=35)

        text1_label = tk.Label(self.root, text=subtitle, font=("Arial", 18, "bold"))
        text1_label.place(x=230, y=83)

        canvas = tk.Canvas(self.root, width=20, height=180)
        canvas.place(x=210, y=120)
        canvas.create_line(10, 10, 10, 170, fill="green", width=1)

        self.enhancer.apply({
            "sidebars": [left_frame],
            "accent":   [name_label],
            "muted":    [text1_label],
            "dividers": [canvas],
        })
        image_label.configure(bg=self.enhancer.BG)
        return left_frame, name_label, text1_label, canvas


    def show_home(self):
        self.clear_page()

        self.build_header()

        text2 = tk.Label(self.root, text="Welcome to ECO Track!", font=("Arial", 10))
        text2.place(x=230, y=140)
        text3 = tk.Label(self.root, text="Log recycled electronic parts online",font=("Arial", 10))
        text3.place(x=230, y=175)
        text4 = tk.Label(self.root, text="Contribute to a cleaner tomorrow!",font=("Arial", 10))
        text4.place(x=230, y=210)

        continue_ = tk.Button(self.root, text="Continue", bg="green", fg="white",cursor="hand2",pady=3, width=18, command=self.show_login)
        continue_.place(x=370, y=250)

        footer = tk.Label(self.root, text="EcoTrack © 2026", font=("Arial", 8), fg="gray")
        footer.place(x=240, y=370)
        
        self.enhancer.apply({
            "muted":    [ text2, text3, text4,footer],
            "buttons":  [continue_],
        }, show_quote=True, quote_y=310)

        

    def show_login(self):
        self.clear_page()
        left_frame, name_label, text1_label, canvas = self.build_header()

        username_label = tk.Label(self.root, text="Username:", font=("Arial", 10))
        username_label.place(x=230, y=140)
        username_entry = tk.Entry(self.root, width=40)
        username_entry.place(x=234, y=160)

        password_label = tk.Label(self.root, text="Password:", font=("Arial", 10))
        password_label.place(x=230, y=195)
        password_entry = tk.Entry(self.root, width=40, show="*")
        password_entry.place(x=234, y=215)

        def try_login():
            user = username_entry.get().strip()
            pw = password_entry.get().strip()

            if user == "" or pw == "":
                msgbox.showwarning("Missing Info", "Username and Password cannot be empty")
                return

            result = self.check_login(user, pw)
            if not result:
                row = self.get_user(user)
                if row and row[3]:
                    msgbox.showerror("Failed", f"Invalid password.\n\nHint: {row[3]}")
                else:
                    msgbox.showerror("Failed", "Invalid username or password.")
                return

            self.current_user = user
            self.show_log_item()

        login_button = tk.Button(self.root, text="LOGIN", bg="green", fg="white",cursor="hand2",width=14, command=try_login)
        login_button.place(x=234, y=250)

        signup_button = tk.Button(self.root, text="SIGN UP", width=14,cursor="hand2",command=self.show_sign_up, bg="white", fg="green")
        signup_button.place(x=370, y=250)

        footer = tk.Label(self.root, text="EcoTrack © 2026", font=("Arial", 8), fg="gray")
        footer.place(x=240, y=370)
        
        self.enhancer.apply({
            "muted":     [username_label, password_label, footer],
            "entries":   [username_entry, password_entry],
            "buttons":   [login_button],
            "secondary": [signup_button],
        }, show_quote=True, quote_y=310)

    def show_sign_up(self):
        self.clear_page()
        left_frame, name_label, text1_label, canvas = self.build_header()

        navbar = tk.Frame(
            self.root,
            bg=BG_CARD,
            height=40
        )
        navbar.place(x=0, y=0, width=600, height=40)
    
        login_btn = tk.Button(
            navbar,
            text="Login",
            bg=DANGER,
            fg="white",
            bd=0,
            cursor="hand2",
            padx=10,
            command=self.show_login
        )
        login_btn.pack(side="right", padx=15)
        
        label = tk.Label(self.root, text="Set Username:", font=("Arial", 9))
        label.place(x=230, y=140)
        entry = tk.Entry(self.root, width=40)
        entry.place(x=234, y=160)

        pass_ = tk.Label(self.root, text="Set Password:", font=("Arial", 9))
        pass_.place(x=234, y=195)
        pass_entry = tk.Entry(self.root, width=40, show="*")
        pass_entry.place(x=234, y=215)

        hint = tk.Label(self.root, text="Password Hint:", font=("Arial", 9))
        hint.place(x=234, y=250)
        hint_entry = tk.Entry(self.root, width=40)
        hint_entry.place(x=234, y=270)

        footer = tk.Label(self.root, text="EcoTrack © 2026", font=("Arial", 8), fg="gray")
        footer.place(x=240, y=370)

        def show_hint():
            user = user_entry.get().strip()
            if not user:
                msgbox.showwarning("Missing Info", "Please enter your username.")
                return
            row = self.get_user(user)
            if not row:
                msgbox.showerror("Not Found", "No account found with that username.")
                return
            hint = row[3]
            hint_label.configure(
                text=f"Hint: {hint}" if hint else "No hint was set for this account."
            )

        def signed_up():
            user = entry.get().strip()
            pw = pass_entry.get().strip()
            hint = hint_entry.get().strip()

            if user == "" or pw == "":
                msgbox.showwarning("Missing Info", "Username and Password cannot be empty")
                return

            success = self.save_user(user, pw, hint)
            if not success:
                msgbox.showwarning("Taken", "That username is already registered.")
                return

            msgbox.showinfo("Success", "Account created! Please log in.")
            self.show_login()

        sign_button = tk.Button(self.root, text="Sign Up", bg="white", fg="green",cursor="hand2",width=34, command=signed_up)
        sign_button.place(x=234, y=300)

        self.enhancer.apply({
            "muted":    [ label, pass_,hint, footer],
            "entries":  [entry, pass_entry,hint_entry],
            "buttons":  [sign_button],
        }, show_quote=True, quote_y=335)

    def show_records(self):
        self.clear_page()
        left_frame, name_label, text1_label, canvas = self.build_header()
        navbar = tk.Frame(
            self.root,
            bg=BG_CARD,
            height=40
        )
        navbar.place(x=0, y=0, width=600, height=40)
        
        log_btn = tk.Button(
            navbar,
            text="Log item",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_log_item
        )
        log_btn.pack(side="left", padx=10)
    
        goals_btn = tk.Button(
            navbar,
            text="Goals",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_goals
        )
        goals_btn.pack(side="left", padx=10)
    
        feedback_btn = tk.Button(
            navbar,
            text="Feedback",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_feedback
        )
        feedback_btn.pack(side="left", padx=10)
    
        tips_btn = tk.Button(
            navbar,
            text="Tips",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_tips
        )
        tips_btn.pack(side="left", padx=10)
    
    
        logout_btn = tk.Button(
            navbar,
            text="Logout",
            bg=DANGER,
            fg="white",
            bd=0,
            cursor="hand2",
            padx=10,
            command=self.show_login
        )
        logout_btn.pack(side="right", padx=15)

        self.table()
        footer = tk.Label(self.root, text="EcoTrack © 2026", font=("Arial", 8), fg="gray")
        footer.place(x=240, y=370)
        self.enhancer.apply({
            "muted":    [ footer],
        })
        
    def show_log_item(self):
        self.clear_page()
        left_frame, name_label, text1_label, canvas = self.build_header()
        navbar = tk.Frame(
            self.root,
            bg=BG_CARD,
            height=40
        )
        navbar.place(x=0, y=0, width=600, height=40)
        
        goals_btn = tk.Button(
            navbar,
            text="Goals",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_goals
        )
        goals_btn.pack(side="left", padx=10)
    
        feedback_btn = tk.Button(
            navbar,
            text="Feedback",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_feedback
        )
    
        feedback_btn.pack(side="left", padx=10)
    
        tips_btn = tk.Button(
            navbar,
            text="Tips",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_tips
        )
    
        tips_btn.pack(side="left", padx=10)
    
    
        logout_btn = tk.Button(
            navbar,
            text="Logout",
            bg=DANGER,
            fg="white",
            bd=0,
            cursor="hand2",
            padx=10,
            command=self.show_login
        )
    
        logout_btn.pack(side="right", padx=15)

        
        style = ttk.Style()
        style.theme_use("clam")
        style.configure(
                    "TCombobox",
                    fieldbackground=BG_DARK,
                    background=BG_DARK,
                    foreground="white",
                    bordercolor=ACCENT1,
                    lightcolor=ACCENT1,
                    darkcolor=ACCENT1,
                    arrowcolor="white"
                )
        tk.Label(self.root, text=f"Welcome, {self.current_user}",font=("Arial", 12), bg=BG_DARK,fg=ACCENT1).place(x=230, y=115)

        log_label = tk.Label(self.root, text="Item:", font=("Arial", 10))
        log_label.place(x=230, y=140) 
        item_var = tk.StringVar()
        item_menu = ttk.Combobox(self.root,textvariable=item_var,values=item_types,width=37)
        item_menu.place(x=234,y=165)

        quantity = tk.Label(self.root, text="Quantity:", font=("Arial", 10))
        quantity.place(x=230, y=195)
        quantity_entry = tk.Entry(self.root, width=28)
        quantity_entry.place(x=234, y=215) 
        
        footer = tk.Label(self.root, text="EcoTrack © 2026", font=("Arial", 8), fg="gray")
        footer.place(x=240, y=370)

        
        def log_and_update():
            item = item_menu.get().strip()
            quantity = quantity_entry.get().strip()
        
            if item == "" or quantity == "":
                msgbox.showwarning("Missing Info", "Item and Quantity cannot be empty")
                return
        
            try:
                qty_int = int(quantity)
            except ValueError:
                msgbox.showerror("Invalid Input", "Quantity must be a whole number")
                return
        
            self.save_log(item, qty_int)
        
            goal_messages = self.goal.add(qty_int)

            if goal_messages:
                msgbox.showinfo("Saved", "Item logged successfully!\n\n" + "\n".join(goal_messages))
            else:
                msgbox.showinfo("Saved", "Item logged successfully!")
        
            item_menu.delete(0, tk.END)
            quantity_entry.delete(0, tk.END)
                        
    
        log_button = tk.Button(self.root, text="Log Item", bg="green", fg="white",width=14, cursor="hand2",command=log_and_update)
        log_button.place(x=234, y=250) 
        records_button = tk.Button(root, text="View Records", width=14, cursor="hand2",command=self.show_records,bg="white", fg="green")
        records_button.place(x=370, y=250)

        self.enhancer.apply({
            "muted":    [log_label, quantity, footer],
            "entries":  [ quantity_entry],
            "buttons":  [log_button],
            "secondary": [records_button],
        }, show_quote=True, quote_y=310)
        
    def show_goals(self):
        self.clear_page()
        left_frame, name_label, text1_label, canvas = self.build_header()

        navbar = tk.Frame(
            self.root,
            bg=BG_CARD,
            height=40
        )
        navbar.place(x=0, y=0, width=600, height=40)
        
        log_btn = tk.Button(
            navbar,
            text="Log item",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_log_item
        )
        log_btn.pack(side="left", padx=10)

        record_btn = tk.Button(
            navbar,
            text="Records",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_records
        )
        record_btn.pack(side="left", padx=10)
    
        feedback_btn = tk.Button(
            navbar,
            text="Feedback",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_feedback
        )
        feedback_btn.pack(side="left", padx=10)
    
        tips_btn = tk.Button(
            navbar,
            text="Tips",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_tips  
        )
        tips_btn.pack(side="left", padx=10)
    
    
        logout_btn = tk.Button(
            navbar,
            text="Logout",
            bg=DANGER,
            fg="white",
            bd=0,
            cursor="hand2",
            padx=10,
            command=self.show_login
        )
    
        logout_btn.pack(side="right", padx=15)

        card = tk.Frame(
            self.root,
            bg=BG_CARD,
            highlightbackground=ACCENT1,
            highlightthickness=1
        )
        card.place(x=120, y=130, width=420, height=210)
    
        daily_pct = int((self.goal.today / self.goal.daily) * 100) if self.goal.daily > 0 else 0
        weekly_pct = int((self.goal.today / self.goal.weekly) * 100) if self.goal.weekly > 0 else 0

        name_label = tk.Label(card, text="Keep track of your goals on this device", fg=ACCENT1,bg = BG_CARD,font=("Arial", 11, "bold"))
        name_label.place(x=25, y=2)

        daily = tk.Label(
            card,
            text="DAILY GOAL",
            font=("Arial", 10, "bold"),
            fg=ACCENT2,
            bg = BG_CARD
        )
        daily.place(x=25, y=25)
    
        tk.Label(
            card,
            text=f"{self.goal.today} / {self.goal.daily} items",
            font=("Arial", 9),
            fg=TEXT_MUTE,
            bg=BG_CARD
        ).place(x=25, y=48)
    
        daily_bar_bg = tk.Frame(card, bg=ENTRY_BG)
        daily_bar_bg.place(x=25, y=75, width=300, height=14)
    
        daily_width = min(int((self.goal.today / self.goal.daily) * 300), 300) if self.goal.daily > 0 else 0
    
        tk.Frame(
            card,
            bg=ACCENT1,
            width=daily_width,
            height=14
        ).place(x=25, y=75)
    
        tk.Label(
            card,
            text=f"{daily_pct}%",
            font=("Arial", 9, "bold"),
            fg=ACCENT1,
            bg=BG_CARD
        ).place(x=345, y=72)

        weekly = tk.Label(
            card,
            text="WEEKLY GOAL",
            font=("Arial", 10, "bold"),
            fg=ACCENT2,
            bg=BG_CARD
        )
        weekly.place(x=25, y=115)
    
        tk.Label(
            card,
            text=f"{self.goal.today} / {self.goal.weekly} items",
            font=("Arial", 9),
            fg=TEXT_MUTE,
            bg=BG_CARD
        ).place(x=25, y=138)
    
        weekly_bar_bg = tk.Frame(card, bg=ENTRY_BG)
        weekly_bar_bg.place(x=25, y=165, width=300, height=14)
    
        weekly_width = min(int((self.goal.today / self.goal.weekly) * 300), 300) if self.goal.weekly > 0 else 0
    
        tk.Frame(
            card,
            bg=ACCENT2,
            width=weekly_width,
            height=14
        ).place(x=25, y=165)
    
        tk.Label(
            card,
            text=f"{weekly_pct}%",
            font=("Arial", 9, "bold"),
            fg=ACCENT2,
            bg=BG_CARD
        ).place(x=345, y=162)
    
        total_card = tk.Frame(
            self.root,
            bg=BG_CARD,
            highlightbackground=ACCENT1,
            highlightthickness=1
        )
        total_card.place(x=120, y=350, width=200, height=35)
    
        tk.Label(
            total_card,
            text=f"TOTAL: {self.goal.total} items",
            font=("Arial", 10, "bold"),
            fg=TEXT_MAIN,
            bg=BG_CARD
        ).place(x=25, y=7)

        btn = tk.Button(
            self.root,
            text="SET NEW GOALS",
            bg=ACCENT1,
            fg=BG,
            font=("Arial", 9, "bold"),
            width=18,
            bd=0,
            cursor="hand2",
            activebackground=ACCENT4,
            command=self.set_goals_dialog
        )
        btn.place(x=410, y=350)

    def set_goals_dialog(self):
        d = tk.Toplevel(self.root)
        d.title("Set Goals")
        d.geometry("280x180")
        d.configure(bg=self.enhancer.BG)

        tk.Label(d, text="Daily Goal:", bg=self.enhancer.BG,fg=self.enhancer.TEXT).pack(pady=8)
        daily_e = tk.Entry(d)
        daily_e.insert(0, str(self.goal.daily))
        daily_e.pack()

        tk.Label(d, text="Weekly Goal:", bg=self.enhancer.BG,fg=self.enhancer.TEXT).pack(pady=8)
        weekly_e = tk.Entry(d)
        weekly_e.insert(0, str(self.goal.weekly))
        weekly_e.pack()

        def save():
            try:
                nd, nw = int(daily_e.get()), int(weekly_e.get())
                if nd > 0 and nw > 0:
                    self.goal.set_daily(nd)
                    self.goal.set_weekly(nw)
                    msgbox.showinfo("Success", "Goals updated!")
                    d.destroy()
                    self.show_goals()
                else:
                    msgbox.showwarning("Error", "Enter positive numbers!")
            except:
                msgbox.showwarning("Error", "Enter valid numbers!")

        tk.Button(d, text="Save", bg="green", fg="white", command=save).pack(pady=12)


    def set_rating(self, n):
        self.star_rating.set(n)
        for i, btn in enumerate(self.star_buttons):
            btn.configure(fg=ACCENT2 if i < n else TEXT_MUTE)

    def on_key(self, event=None):
        text = self.feedback_box.get("1.0", tk.END).strip()
        count = len(text)
        if count > 300:
            self.feedback_box.delete("1.0", tk.END)
            self.feedback_box.insert("1.0", text[:300])
            count = 300
        self.char_counter.configure(text=f"{count}/300")

    def submit_feedback(self):
        text = self.feedback_box.get("1.0", tk.END).strip()
        rating = self.star_rating.get()

        if rating == 0:
            msgbox.showwarning("Missing Rating", "Please select a star rating.")
            return
        if not text:
            msgbox.showwarning("Empty Feedback", "Please write some feedback before submitting.")
            return

        msgbox.showinfo("Successful", f"Thanks for your {rating}★ review!")
        self.feedback_box.delete("1.0", tk.END)
        self.set_rating(0)
        self.char_counter.configure(text="0/300")

    def show_feedback(self):
        self.clear_page()
        self.star_rating = tk.IntVar(value=0)

        navbar = tk.Frame(
            self.root,
            bg=BG_CARD,
            height=40
        )
        navbar.place(x=0, y=0, width=600, height=40)
        
        log_btn = tk.Button(
            navbar,
            text="Log item",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_log_item
        )
        log_btn.pack(side="left", padx=10)

        record_btn = tk.Button(
            navbar,
            text="Records",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_records
        )
    
        record_btn.pack(side="left", padx=10)
    
        goals_btn = tk.Button(
            navbar,
            text="Goals",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_goals
        )
    
        goals_btn.pack(side="left", padx=10)
    
        tips_btn = tk.Button(
            navbar,
            text="Tips",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_tips
            
        )
    
        tips_btn.pack(side="left", padx=10)
    
        logout_btn = tk.Button(
            navbar,
            text="Logout",
            bg=DANGER,
            fg="white",
            bd=0,
            cursor="hand2",
            padx=10,
            command=self.show_login
        )
        logout_btn.pack(side="right", padx=15)


        card = tk.Frame(self.root, bg=BG_CARD, padx=15, pady=15)
        card.place(x=30, y=45, width=540, height=340)  

        tk.Label(card, font=('Arial', 12, 'bold'), text='How is the app?',
                  bg=BG_CARD, fg=TEXT_MAIN).pack(anchor="w")
        tk.Label(card, font=('Arial', 9), text='We value your feedback.',
                  bg=BG_CARD, fg=TEXT_MUTE).pack(anchor="w", pady=(0, 8))

        star_frame = tk.Frame(card, bg=BG_CARD)
        star_frame.pack(anchor="w", pady=(0, 8))

        tk.Label(star_frame, text="Rating: ", bg=BG_CARD, fg=TEXT_MAIN,
                  font=("Arial", 10)).pack(side=tk.LEFT)

        for i in range(1, 6):
            btn = tk.Button(star_frame, text="★", font=("Arial", 14), bg=BG_CARD,fg=TEXT_MUTE, bd=0, cursor="hand2",command=lambda n=i: self.set_rating(n))
            btn.pack(side=tk.LEFT)
            self.star_buttons.append(btn)

        self.feedback_box = tk.Text(card, font=('Arial', 9), height=6, fg="white",
                                     bg=BG_DARK, insertbackground="white", relief="flat")
        self.feedback_box.pack(fill=tk.X, pady=(0, 4))
        self.feedback_box.bind("<KeyRelease>", self.on_key)

        self.char_counter = tk.Label(card, text="0/300", bg=BG_CARD, fg=TEXT_MUTE,
                                      font=("Arial", 8))
        self.char_counter.pack(anchor="e")

        tk.Button(card, font=("Arial", 10, "bold"), text="Submit", bg=ACCENT1,fg="black", command=self.submit_feedback, cursor="hand2",relief="flat", padx=15, pady=4).pack(anchor="e", pady=(8, 0))

    def show_tips(self):

        self.clear_page()
        left_frame, name_label, text1_label, canvas = self.build_header()
        
        navbar = tk.Frame(
            self.root,
            bg=BG_CARD,
            height=40
        )
        navbar.place(x=0, y=0, width=600, height=40)
        
        log_btn = tk.Button(
            navbar,
            text="Log item",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_log_item
        )
        log_btn.pack(side="left", padx=10)
    
        goals_btn = tk.Button(
            navbar,
            text="Goals",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_goals
        )
    
        goals_btn.pack(side="left", padx=10)
    
        feedback_btn = tk.Button(
            navbar,
            text="Feedback",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_feedback
        )
    
        feedback_btn.pack(side="left", padx=10)
    
        record_btn = tk.Button(
            navbar,
            text="Records",
            bg=BG_CARD,
            fg=TEXT_MAIN,
            bd=0,
            cursor="hand2",
            activebackground=BG_CARD,
            activeforeground=ACCENT1,
            command=self.show_records
        )
    
        record_btn.pack(side="left", padx=10)
    
    
        logout_btn = tk.Button(
            navbar,
            text="Logout",
            bg=DANGER,
            fg="white",
            bd=0,
            cursor="hand2",
            padx=10,
            command=self.show_login
        )

        card = tk.Frame(
            self.root,
            bg=BG_CARD,
            highlightbackground=ACCENT1,
            highlightthickness=1
        )
    
        card.place(x=140, y=130, width=390, height=240)
    
        tip1 = tk.Label(card,text="Don't trash it, recycle it:\nOld phones, cables, and batteries contain toxic materials.\nAlways drop them at certified e-waste points.",justify="left",
            font=("Arial", 9),padx=10,pady=10,anchor="w"
        )
        tip1.pack(fill="x", pady=4)
    
    
        tip2 = tk.Label(card,text="Battery safety:\nNever throw batteries into regular bins.\nThey can leak acid or even cause fires.",justify="left",
            font=("Arial", 9),padx=10,pady=10,anchor="w"
        )
    
        tip2.pack(fill="x", pady=4)
    
        tip3 = tk.Label(card,text="Phones have value:\nSmartphones contain gold, copper,\nand rare-earth metals that can be recovered.",justify="left",
            font=("Arial", 9),padx=10,pady=10,anchor="w"
        )
    
        tip3.pack(fill="x", pady=4)
    
    
        footer = tk.Label(self.root,text="EcoTrack © 2026",font=("Arial", 8),fg="gray")
        footer.place(x=250, y=370)
    
    
        self.enhancer.apply({
            "muted": [tip1, tip2, tip3,footer]
        })

root = tk.Tk()
app = EcoTrack(root)
root.mainloop()